In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 使用 Keras API 加载数据
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

learning_rate = 1e-4
keep_prob_rate = 0.7 #
dropout_rate = 1 - keep_prob_rate 
max_epoch = 2000
batch_size = 100 

def preprocess(x, y):
    x = tf.cast(x, tf.float32) / 255.0
    x = tf.reshape(x, [28, 28, 1])
    y = tf.one_hot(tf.cast(y, tf.int32), depth=10)
    return x, y

train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_dataset = train_dataset.map(preprocess).shuffle(60000).batch(batch_size).repeat()
train_iterator = iter(train_dataset)

x_test_processed = tf.reshape(tf.cast(x_test, tf.float32) / 255.0, [-1, 28, 28, 1])
y_test_processed = tf.one_hot(tf.cast(y_test, tf.int32), depth=10)


class CNN(models.Model):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = layers.Conv2D(32, kernel_size=7, padding='same', activation='relu')
        self.pool1 = layers.MaxPool2D(pool_size=2, strides=2, padding='same')

        self.conv2 = layers.Conv2D(64, kernel_size=5, padding='same', activation='relu')
        self.pool2 = layers.MaxPool2D(pool_size=2, strides=2, padding='same')

        self.flatten = layers.Flatten()
        self.fc1 = layers.Dense(1024, activation='relu')
        self.dropout = layers.Dropout(rate=dropout_rate)

        self.fc2 = layers.Dense(10, activation='softmax')

    def call(self, inputs, training=False):
        x = self.conv1(inputs)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.pool2(x)
        x = self.flatten(x)
        x = self.fc1(x)
        if training:
            x = self.dropout(x)
        return self.fc2(x)

model = CNN()

cross_entropy = tf.keras.losses.CategoricalCrossentropy()
optimizer = tf.keras.optimizers.Adam(learning_rate)
accuracy_metric = tf.keras.metrics.CategoricalAccuracy()


for i in range(max_epoch):
    batch_xs, batch_ys = next(train_iterator)

    with tf.GradientTape() as tape:
        predictions = model(batch_xs, training=True)
        loss = cross_entropy(batch_ys, predictions)

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    if i % 100 == 0:

        accuracy_metric.reset_state()
        test_predictions = model(x_test_processed[:1000], training=False)
        accuracy_metric.update_state(y_test_processed[:1000], test_predictions)
        accuracy_result = accuracy_metric.result()
        print(f"第 {i} 步, 测试准确率: {accuracy_result.numpy():.4f}")

第 0 步, 测试准确率: 0.1340
第 100 步, 测试准确率: 0.8570
第 200 步, 测试准确率: 0.9210
第 300 步, 测试准确率: 0.9350
第 400 步, 测试准确率: 0.9480
第 500 步, 测试准确率: 0.9550
第 600 步, 测试准确率: 0.9590
第 700 步, 测试准确率: 0.9690
第 800 步, 测试准确率: 0.9750
第 900 步, 测试准确率: 0.9720
第 1000 步, 测试准确率: 0.9760
第 1100 步, 测试准确率: 0.9820
第 1200 步, 测试准确率: 0.9810
第 1300 步, 测试准确率: 0.9830
第 1400 步, 测试准确率: 0.9820
第 1500 步, 测试准确率: 0.9820
第 1600 步, 测试准确率: 0.9820
第 1700 步, 测试准确率: 0.9860
第 1800 步, 测试准确率: 0.9770
第 1900 步, 测试准确率: 0.9850
